# Calculate unit-wise difference between the Taunton-Worcester scenario and base

Aggregate differences by stroke unit.

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
dir_data = 'data'
dir_output = 'output'
# Scenario results:
file_results_comparison = 'results_full_comparison_taunworc_base.csv'
# Reference:
file_hospitals = 'stroke_hospitals_2025.csv'
# Output:

## Load data

### Scenario comparison results

In [3]:
p = os.path.join(dir_output, file_results_comparison)
df = pd.read_csv(p, index_col=0)

In [4]:
display(df.head(3))

,area_lower,Admissions_2025_Est,nearest_ivt_unit_base,transfer_unit_base,transfer_required_base,nearest_ivt_unit_no_bias_base,drip_ship_mt_travel_time_base,drip_ship_mrs_0-2_base,lvo_drip_ship_ivt_mt_mrs_0-2_base,drip_ship_transfer0!6_mrs_0-2_base,...,drip_ship_mrs_0-2_taunworc,lvo_drip_ship_ivt_mt_mrs_0-2_taunworc,drip_ship_transfer0!6_mrs_0-2_taunworc,lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_taunworc,change_in_drip_ship_mt_travel_time,change_in_drip_ship_mrs_0-2,change_in_lvo_drip_ship_ivt_mt_mrs_0-2,change_in_drip_ship_transfer0!6_mrs_0-2,change_in_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2,changed
LSOA,,,,,,,,,,,,,,,,,,,,,
ADUR001A,Adur 001A,1.49403,BN25BE,BN25BE,False,BN25BE,10.9,0.619,0.429,0.619,...,0.619,0.429,0.619,0.429,0.0,0.0,0.0,0.0,0.0,False
ADUR001B,Adur 001B,1.86762,BN25BE,BN25BE,False,BN25BE,11.9,0.619,0.429,0.619,...,0.619,0.429,0.619,0.429,0.0,0.0,0.0,0.0,0.0,False
ADUR001C,Adur 001C,2.24110,BN112DH,BN25BE,True,BN112DH,95.0,0.603,0.376,0.597,...,0.603,0.376,0.597,0.356,0.0,0.0,0.0,0.0,0.0,False


In [5]:
print(df.columns)

Index(['area_lower', 'Admissions_2025_Est', 'nearest_ivt_unit_base',
       'transfer_unit_base', 'transfer_required_base',
       'nearest_ivt_unit_no_bias_base', 'drip_ship_mt_travel_time_base',
       'drip_ship_mrs_0-2_base', 'lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'drip_ship_transfer0!6_mrs_0-2_base',
       'lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_base',
       'nearest_ivt_unit_taunworc', 'transfer_unit_taunworc',
       'transfer_required_taunworc', 'nearest_ivt_unit_no_bias_taunworc',
       'drip_ship_mt_travel_time_taunworc', 'drip_ship_mrs_0-2_taunworc',
       'lvo_drip_ship_ivt_mt_mrs_0-2_taunworc',
       'drip_ship_transfer0!6_mrs_0-2_taunworc',
       'lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_taunworc',
       'change_in_drip_ship_mt_travel_time', 'change_in_drip_ship_mrs_0-2',
       'change_in_lvo_drip_ship_ivt_mt_mrs_0-2',
       'change_in_drip_ship_transfer0!6_mrs_0-2',
       'change_in_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2', 'changed'],
      dtype='str'

Gather columns by type and without tailing scenario names (`_base`, `_taunworc`):

In [6]:
col_lsoa11nm = 'area_lower'
col_admissions = 'Admissions_2025_Est'
cols_unit_postcodes = ['nearest_ivt_unit', 'transfer_unit',
                       'nearest_ivt_unit_no_bias']
cols_transfer = ['transfer_required']
cols_travel_time = ['drip_ship_mt_travel_time']
cols_outcome = [
    'drip_ship_mrs_0-2',
    'lvo_drip_ship_ivt_mt_mrs_0-2',
    'drip_ship_transfer0!6_mrs_0-2',
    'lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2',
]
cols_change = [
    'change_in_drip_ship_mt_travel_time',
    'change_in_drip_ship_mrs_0-2',
    'change_in_lvo_drip_ship_ivt_mt_mrs_0-2',
    'change_in_drip_ship_transfer0!6_mrs_0-2',
    'change_in_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2',
    'changed'
]

cols_shared = [col_lsoa11nm, col_admissions] + cols_change

### Hospital reference

In [7]:
# Load hospital coordinates from CSV and create a GeoDataFrame
p = os.path.join(dir_data, file_hospitals)
df_hospitals = pd.read_csv(p, index_col='Postcode')

# Only keep hospitals in England:
df_hospitals = df_hospitals.loc[df_hospitals["Country"] == "England"]

In [8]:
display(df_hospitals.head(3))

,SSNAP name,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT
Postcode,,,,,,,,,,,
OX39DU,John Radcliffe Hospital,England,"Buckinghamshire, Oxford and Berkshire",453940,207531,-1.219806,51.763871,Oxford,E01028534,1,1
RG15AN,Royal Berkshire Hospital,England,"Buckinghamshire, Oxford and Berkshire",472411,172962,-0.959330,51.451022,Reading,E01016414,1,0
HP112TT,Wycombe General Hospital,England,"Buckinghamshire, Oxford and Berkshire",486386,192694,-0.753428,51.626442,High Wycombe,E01017839,1,0


In [9]:
print(len(df_hospitals))

131


## Define functions

Create some functions for the calculations.

In [10]:
def append_scenario_to_column_names(cols, scens):
    new_cols = [f'{c}_{s}' for c in cols for s in scens]
    return new_cols

def remove_scenario_from_column_names(cols, scens):
    new_cols = [c.replace(f'_{s}', '').replace(s, '')
                for c in cols for s in scens]
    return new_cols

In [46]:
def calculate_admissions_by_unit(
        df,
        col_admissions,
        cols_unit_postcodes,
        transfer_rates=[1.0]
    ):
    df_admissions = pd.concat(
        [df.groupby(col)[col_admissions].sum().rename(f'admissions_{col}')
         for col in cols_unit_postcodes],
        axis='columns'
    )
    # Calculate transfers:
    m = df_admissions['admissions_transfer_unit'].notna()
    for t in transfer_rates:
        p = '' if t == 1.0 else f'_transfer{t}'.replace('.', '!')
        df_admissions[f'admissions_transfers{p}'] = (
            df_admissions['admissions_nearest_ivt_unit'].copy() * t)
        df_admissions.loc[m, f'admissions_transfers{p}'] = pd.NA
        # Repeat for no travel time bias:
        df_admissions[f'admissions_transfers{p}_no_bias'] = (
            df_admissions['admissions_nearest_ivt_unit_no_bias'].copy() * t)
        df_admissions.loc[m, f'admissions_transfers{p}_no_bias'] = pd.NA

    df_admissions = df_admissions.fillna(0.0)
    df_admissions = df_admissions.round(1)
    return df_admissions

In [31]:
def calculate_numbers_of_thrombectomy(df_admissions, mt_rate):
    df_mt = df_admissions * mt_rate
    df_mt = df_mt.rename(
        columns=dict([(c, f'mt_{c}') for c in df_mt.columns]))

    df_mt = df_mt.round(1)
    return df_mt

In [13]:
def calculate_weighted_outcomes_by_lsoa(df, col_admissions, cols_outcome):
    """LSOA-level"""
    list_weighted_outcomes = []
    for col in cols_outcome:
        s = df[col] * df[col_admissions].rename(f'admissions_weighted_{col}')
        list_weighted_outcomes.append(s)

    df_outcomes = pd.concat(list_weighted_outcomes, axis='columns')
    df_outcomes = df_outcomes.round(5)
    return df_outcomes

In [96]:
def calculate_outcomes_by_unit(
        df_outcomes,  # by LSOA
        df_admissions,  # by unit
        cols_unit_postcodes,
        cols_admissions_weighted_outcomes
    ):
    list_dfs_unit_outcomes = []
    for col in cols_unit_postcodes:
        df_unit_outcomes = df_outcomes[
            [col] + cols_admissions_weighted_outcomes].groupby(col).sum()
        rename_dict = dict([(c, f'{col}_{c}')
                            for c in df_unit_outcomes.columns])
        df_unit_outcomes = df_unit_outcomes.rename(columns=rename_dict)
        list_dfs_unit_outcomes.append(df_unit_outcomes)
    df_unit_outcomes = pd.concat(list_dfs_unit_outcomes, axis='columns')

    
    # Divide outcomes by the direct admissions to each unit:
    def scale_outcomes_by_admissions(df_unit_outcomes, df_admissions, col):
        for unit in df_admissions.index:
            df_unit_outcomes.loc[unit] = (
                df_unit_outcomes.loc[unit] /
                df_admissions.loc[unit, col]
            )
        return df_unit_outcomes

    d1 = scale_outcomes_by_admissions(
        df_unit_outcomes.copy(),
        df_admissions,
        'admissions_nearest_ivt_unit'
    )
    rename_dict = dict([(c, f'{c}_no_bias')
                        for c in df_unit_outcomes.columns])
    d2 = scale_outcomes_by_admissions(
        df_unit_outcomes.copy(),
        df_admissions,
        'admissions_nearest_ivt_unit_no_bias'
    ).rename(columns=rename_dict)
    df_unit_outcomes = pd.concat((d1, d2), axis='columns')

    df_unit_outcomes = df_unit_outcomes.round(5)
    return df_unit_outcomes

Gather these separate functions into a main one:

In [97]:
def main(
        df,
        scenario,
        mt_rate,
        transfer_rates,
        cols_shared,
        col_admissions,
        cols_unit_postcodes,
        cols_outcome,
    ):
    """
    """
    df_here = df[cols_shared + [c for c in df.columns if c.endswith(scenario)]]
    rename_dict = dict([(c, c.replace(f'_{scenario}', ''))
                        for c in df_here.columns])
    df_here = df_here.rename(columns=rename_dict)
    
    # Sum admissions to each unit by scenario
    df_admissions = calculate_admissions_by_unit(
        df_here, col_admissions, cols_unit_postcodes, transfer_rates)
    
    # Calculate numbers of thrombectomy
    # Scale down the total admissions to reflect that not everybody
    # receives MT.
    df_mt = calculate_numbers_of_thrombectomy(df_admissions, mt_rate)
    
    # Admissions-weighted outcomes
    # First calculate admissions-weighted outcomes by LSOA:
    df_outcomes = calculate_weighted_outcomes_by_lsoa(
        df_here, col_admissions, cols_outcome)
    cols_admissions_weighted_outcomes = list(df_outcomes.columns)
    # Merge in unit info and admissions:
    df_outcomes = pd.concat(
        (df_here[cols_unit_postcodes], df_outcomes),
        axis='columns'
    )

    # Group by unit:
    df_unit_outcomes = calculate_outcomes_by_unit(
        df_outcomes,  # by LSOA
        df_admissions,  # by unit
        cols_unit_postcodes,
        cols_admissions_weighted_outcomes
    )

    # Gather all results:
    comparison_by_ivt_unit = pd.concat(
        (df_admissions, df_mt, df_unit_outcomes),
        axis='columns'
    )
    return comparison_by_ivt_unit

## Set up scenarios

Run calculations separately for each scenario, otherwise it's difficult to see what's happening.

In [98]:
mt_rate = 0.1
transfer_rates = [1.0, 0.6]  # should match existing results files.

comparison_by_ivt_unit_base = main(
    df,
    'base',
    mt_rate,
    transfer_rates,
    cols_shared,
    col_admissions,
    cols_unit_postcodes,
    cols_outcome,
)

comparison_by_ivt_unit_taunworc = main(
    df,
    'taunworc',
    mt_rate,
    transfer_rates,
    cols_shared,
    col_admissions,
    cols_unit_postcodes,
    cols_outcome,
)

## Check for changes between scenarios

In [99]:
m = (comparison_by_ivt_unit_base.fillna(0.0) ==
     comparison_by_ivt_unit_taunworc.fillna(0.0)).all(axis='columns')

units_with_change = comparison_by_ivt_unit_base.loc[~m].index

In [100]:
display(df_hospitals.loc[
    units_with_change, ['SSNAP name', 'Hospital_City']])

,SSNAP name,Hospital_City
B152TH,Queen Elizabeth Hospital Edgbaston,Birmingham
BS105NB,Bristol and Weston SU,North Bristol
DT12JY,Dorset County Hospital,Dorchester
DY12HQ,Russells Hall Hospital,Dudley
EX25DW,Royal Devon and Exeter Hospital,Exeter
EX314JB,North Devon District Hospital,Barnstaple
GL537AN,Cheltenham & Gloucester Hospitals,Cheltenham
HR12ER,Hereford County Hospital,Hereford
PL68DH,Derriford Hospital,Plymouth
SO166YD,Southampton General Hospital,Southampton


See that the expected units have changed - ones near Taunton and Worcester.

## Gather and simplify info

Place all results in same dataframe:

In [101]:
# Merge in hospital info:
units = sorted(list(
    set(comparison_by_ivt_unit_base.index) |
    set(comparison_by_ivt_unit_taunworc.index)
))
df_ivt_units = df_hospitals.loc[
    units, ['SSNAP name', 'Hospital_City']].copy()

rename_base = dict([(c, f'{c}_base')
                    for c in comparison_by_ivt_unit_base.columns])
rename_taunworc = dict([(c, f'{c}_taunworc')
                       for c in comparison_by_ivt_unit_taunworc.columns])
comparison_by_ivt_unit = pd.concat(
    (df_ivt_units,
     comparison_by_ivt_unit_base.rename(columns=rename_base),
     comparison_by_ivt_unit_taunworc.rename(columns=rename_taunworc)),
    axis='columns'
)

In [102]:
comparison_by_ivt_unit

,SSNAP name,Hospital_City,admissions_nearest_ivt_unit_base,admissions_transfer_unit_base,admissions_nearest_ivt_unit_no_bias_base,admissions_transfers_base,admissions_transfers_no_bias_base,admissions_transfers_transfer0!6_base,admissions_transfers_transfer0!6_no_bias_base,mt_admissions_nearest_ivt_unit_base,...,nearest_ivt_unit_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_no_bias_taunworc,nearest_ivt_unit_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_no_bias_taunworc,transfer_unit_admissions_weighted_drip_ship_mrs_0-2_no_bias_taunworc,transfer_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_no_bias_taunworc,transfer_unit_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_no_bias_taunworc,transfer_unit_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_no_bias_taunworc,nearest_ivt_unit_no_bias_admissions_weighted_drip_ship_mrs_0-2_no_bias_taunworc,nearest_ivt_unit_no_bias_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_no_bias_taunworc,nearest_ivt_unit_no_bias_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_no_bias_taunworc,nearest_ivt_unit_no_bias_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_no_bias_taunworc
B152TH,Queen Elizabeth Hospital Edgbaston,Birmingham,861.1,7093.9,817.1,0.0,0.0,0.0,0.0,86.1,...,0.65163,0.45131,4.35251,2.77338,4.31305,2.64073,0.61837,0.42834,0.61837,0.42834
B662QT,Midland Metropolitan University Hospital (MMUH...,Smethwick,1799.4,0.0,1865.9,1799.4,1865.9,1079.7,1119.6,179.9,...,0.57700,0.34860,NaN,NaN,NaN,NaN,0.60594,0.38735,0.59893,0.36372
BA13NG,Royal United Hospital Bath,Bath,853.3,0.0,853.3,853.3,853.3,512.0,512.0,85.3,...,0.59291,0.35085,NaN,NaN,NaN,NaN,0.59851,0.36980,0.59291,0.35085
BB23HH,Royal Blackburn Hospital,Blackburn,810.1,0.0,837.8,810.1,837.8,486.1,502.7,81.0,...,0.57700,0.34462,NaN,NaN,NaN,NaN,0.60328,0.37919,0.59735,0.35866
BD96RJ,Bradford and Airedale SU,Bradford,1024.9,0.0,1053.6,1024.9,1053.6,614.9,632.2,102.5,...,0.58115,0.34709,NaN,NaN,NaN,NaN,0.60420,0.37947,0.59800,0.35874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WD180HB,Watford General Hospital,Watford,1208.1,0.0,1256.4,1208.1,1256.4,724.9,753.9,120.8,...,0.57187,0.33821,NaN,NaN,NaN,NaN,0.60092,0.37230,0.59542,0.35438
WF14DG,Pinderfields Hospital,Wakefield,1065.7,0.0,1065.7,1065.7,1065.7,639.4,639.4,106.6,...,0.59716,0.35752,NaN,NaN,NaN,NaN,0.60395,0.37963,0.59716,0.35752
WR51DD,Worcestershire Royal Hospital,Worcester,700.6,0.0,700.6,700.6,700.6,420.3,420.3,70.1,...,0.62163,0.43000,1.74727,1.12383,1.73700,1.08919,0.61738,0.42709,0.61738,0.42709
WV100QP,New Cross Hospital,Wolverhampton,1258.8,0.0,1282.7,1258.8,1282.7,755.3,769.6,125.9,...,0.58558,0.34901,NaN,NaN,NaN,NaN,0.60297,0.37711,0.59700,0.35685


In [103]:
comparison_by_ivt_unit.columns

Index(['SSNAP name', 'Hospital_City', 'admissions_nearest_ivt_unit_base',
       'admissions_transfer_unit_base',
       'admissions_nearest_ivt_unit_no_bias_base', 'admissions_transfers_base',
       'admissions_transfers_no_bias_base',
       'admissions_transfers_transfer0!6_base',
       'admissions_transfers_transfer0!6_no_bias_base',
       'mt_admissions_nearest_ivt_unit_base',
       'mt_admissions_transfer_unit_base',
       'mt_admissions_nearest_ivt_unit_no_bias_base',
       'mt_admissions_transfers_base', 'mt_admissions_transfers_no_bias_base',
       'mt_admissions_transfers_transfer0!6_base',
       'mt_admissions_transfers_transfer0!6_no_bias_base',
       'nearest_ivt_unit_admissions_weighted_drip_ship_mrs_0-2_base',
       'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'nearest_ivt_unit_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_base',
       'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_base',


In [ ]:
cols_target = ['SSNAP_name', 'Hospital_City', 'base_first_admissions',
               'comparator_first_admissions', 'base_mt_unit',
               'comparator_mt_unit', 'base_mt_city', 'comparator_mt_city',
               'base_all_MT_admissions', 'comparator_all_MT_admissions',
               'base_transfer_in_MT', 'comparator_transfer_in_MT',
               'base_outcome_lvo_mrs_0-2', 'comparator_outcome_lvo_mrs_0-2',
               'change_in_first_admissions', 'change_in_all_MT_admissions',
               'change_in_outcome_lvo_mrs_0-2',
               'mt_unit_admissions_changed_by_any_ivt_unit
              ]

In [ ]:
rename_dict = {
    'SSNAP name',
    'Hospital_City',
    'admissions_nearest_ivt_unit_base',
    'admissions_transfer_unit_base',
    'admissions_nearest_ivt_unit_no_bias_base',
    'admissions_transfers_base',
    'admissions_transfers_no_bias_base',
    'admissions_transfers_transfer0!6_base',
    'admissions_transfers_transfer0!6_no_bias_base',
    'mt_admissions_nearest_ivt_unit_base',
    'mt_admissions_transfer_unit_base',
    'mt_admissions_nearest_ivt_unit_no_bias_base',
    'mt_admissions_transfers_base',
    'mt_admissions_transfers_no_bias_base',
    'mt_admissions_transfers_transfer0!6_base',
    'mt_admissions_transfers_transfer0!6_no_bias_base',
    'nearest_ivt_unit_admissions_weighted_drip_ship_mrs_0-2_base',
    'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base',
    'nearest_ivt_unit_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_base',
    'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_base',
    'transfer_unit_admissions_weighted_drip_ship_mrs_0-2_base',
    'transfer_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base',
    'transfer_unit_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_base',
    'transfer_unit_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_base',
    'nearest_ivt_unit_no_bias_admissions_weighted_drip_ship_mrs_0-2_base',
    'nearest_ivt_unit_no_bias_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base',
    'nearest_ivt_unit_no_bias_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_base',
    'nearest_ivt_unit_no_bias_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_base',
    'nearest_ivt_unit_admissions_weighted_drip_ship_mrs_0-2_no_bias_base',
    'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_no_bias_base',
    'nearest_ivt_unit_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_no_bias_base',
    'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_no_bias_base',
    'transfer_unit_admissions_weighted_drip_ship_mrs_0-2_no_bias_base',
    'transfer_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_no_bias_base',
    'transfer_unit_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_no_bias_base',
    'transfer_unit_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_no_bias_base',
    'nearest_ivt_unit_no_bias_admissions_weighted_drip_ship_mrs_0-2_no_bias_base',
    'nearest_ivt_unit_no_bias_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_no_bias_base',
    'nearest_ivt_unit_no_bias_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_no_bias_base',
    'nearest_ivt_unit_no_bias_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_no_bias_base',
    'admissions_nearest_ivt_unit_taunworc',
    'admissions_transfer_unit_taunworc',
    'admissions_nearest_ivt_unit_no_bias_taunworc',
    'admissions_transfers_taunworc',
    'admissions_transfers_no_bias_taunworc',
    'admissions_transfers_transfer0!6_taunworc',
    'admissions_transfers_transfer0!6_no_bias_taunworc',
    'mt_admissions_nearest_ivt_unit_taunworc',
    'mt_admissions_transfer_unit_taunworc',
    'mt_admissions_nearest_ivt_unit_no_bias_taunworc',
    'mt_admissions_transfers_taunworc',
    'mt_admissions_transfers_no_bias_taunworc',
    'mt_admissions_transfers_transfer0!6_taunworc',
    'mt_admissions_transfers_transfer0!6_no_bias_taunworc',
    'nearest_ivt_unit_admissions_weighted_drip_ship_mrs_0-2_taunworc',
    'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_taunworc',
    'nearest_ivt_unit_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_taunworc',
    'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_taunworc',
    'transfer_unit_admissions_weighted_drip_ship_mrs_0-2_taunworc',
    'transfer_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_taunworc',
    'transfer_unit_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_taunworc',
    'transfer_unit_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_taunworc',
    'nearest_ivt_unit_no_bias_admissions_weighted_drip_ship_mrs_0-2_taunworc',
    'nearest_ivt_unit_no_bias_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_taunworc',
    'nearest_ivt_unit_no_bias_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_taunworc',
    'nearest_ivt_unit_no_bias_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_taunworc',
    'nearest_ivt_unit_admissions_weighted_drip_ship_mrs_0-2_no_bias_taunworc',
    'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_no_bias_taunworc',
    'nearest_ivt_unit_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_no_bias_taunworc',
    'nearest_ivt_unit_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_no_bias_taunworc',
    'transfer_unit_admissions_weighted_drip_ship_mrs_0-2_no_bias_taunworc',
    'transfer_unit_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_no_bias_taunworc',
    'transfer_unit_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_no_bias_taunworc',
    'transfer_unit_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_no_bias_taunworc',
    'nearest_ivt_unit_no_bias_admissions_weighted_drip_ship_mrs_0-2_no_bias_taunworc',
    'nearest_ivt_unit_no_bias_admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_no_bias_taunworc',
    'nearest_ivt_unit_no_bias_admissions_weighted_drip_ship_transfer0!6_mrs_0-2_no_bias_taunworc',
    'nearest_ivt_unit_no_bias_admissions_weighted_lvo_drip_ship_transfer0!6_ivt_mt_mrs_0-2_no_bias_taunworc'
}

Some extra calculations:

In [ ]:
# Changes between scenarios
comparison_by_ivt_unit['change_in_first_admissions'] = (
    comparison_by_ivt_unit['comparator_first_admissions']
    - comparison_by_ivt_unit['base_first_admissions']
)
comparison_by_ivt_unit['change_in_all_MT_admissions'] = (
    comparison_by_ivt_unit['comparator_all_MT_admissions']
    - comparison_by_ivt_unit['base_all_MT_admissions']
)
comparison_by_ivt_unit['change_in_outcome_lvo_mrs_0-2'] = (
    comparison_by_ivt_unit['comparator_outcome_lvo_mrs_0-2']
    - comparison_by_ivt_unit['base_outcome_lvo_mrs_0-2']
)